# OpenAI judge with stored HarmBench prompts

Re-evaluate saved classifier prompts verbatim and compare their labels with the
existing HarmBench labels. This notebook calls a paid API. Configure the model,
input file, and API key before running the judgment cells.
The original model name is retained as the default for provenance; set
`OPENAI_JUDGE_MODEL` to a model available to your API account.


## Repository paths

Run this setup cell first. Inputs, outputs, and runtime caches use separate repository directories.


In [ ]:
from pathlib import Path
import os

# Launch Jupyter from the repository, or set INTENT_REPO_ROOT explicitly.
_start = Path(os.environ.get("INTENT_REPO_ROOT", Path.cwd())).expanduser().resolve()
REPO_ROOT = next(
    (p for p in (_start, *_start.parents)
     if (p / "data/artifact_manifest.json").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Clone/extract the repository and set INTENT_REPO_ROOT to its root.")
DATA_DIR = Path(os.environ.get("INTENT_DATA_DIR", REPO_ROOT / "data")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("INTENT_OUTPUT_DIR", REPO_ROOT / "outputs")).expanduser().resolve()
WORK_DIR = Path(os.environ.get("INTENT_WORK_DIR", REPO_ROOT / ".work")).expanduser().resolve()
for _directory in (DATA_DIR, OUTPUT_ROOT, WORK_DIR):
    _directory.mkdir(parents=True, exist_ok=True)

# Colab uploads are optional; local runs use configured input paths.
try:
    from google.colab import files as notebook_files
except ImportError:
    class LocalFiles:
        @staticmethod
        def upload():
            raise FileNotFoundError("Set the notebook input path to an existing local file.")

        @staticmethod
        def download(filename):
            print("Saved:", Path(filename).resolve())

    notebook_files = LocalFiles()


In [ ]:
# Install dependencies if needed.
# In Google Colab, uncomment:
# !pip install -q -U openai pandas openpyxl tqdm


In [ ]:
import os
import re
import time
import random
import pandas as pd

from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI, BadRequestError


In [ ]:
# Set OPENAI_API_KEY in the environment or in Colab Secrets.
# Never place a credential in notebook source.


## API key

Recommended in Google Colab: store a secret named `OPENAI_API_KEY`.

The cell below first checks Colab Secrets, then the `OPENAI_API_KEY` environment variable.


In [ ]:
def get_openai_key():
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            return key
    except Exception:
        pass

    key = os.environ.get("OPENAI_API_KEY")
    if key:
        return key

    raise RuntimeError(
        "OPENAI_API_KEY was not found. Add it to Colab Secrets "
        "or set os.environ['OPENAI_API_KEY']."
    )

client = OpenAI(api_key=get_openai_key())
print("OpenAI client initialized.")


## Configuration

Change `INPUT_FILE` for each result file.

By default, the output keeps the same file type and appends `_gpt_judge`.

If the input already contains judgments made with the **old/custom GPT rubric**, set `OVERWRITE_EXISTING_GPT_JUDGE = True`.


In [ ]:
# =========================
# CONFIG
# =========================

INPUT_FILE = DATA_DIR / "results/qwen3_32b/qwen3_4b_instruct_2507.xlsx"

OUTPUT_DIR = OUTPUT_ROOT / "openai_judge"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / f"{INPUT_FILE.stem}_gpt_judge{INPUT_FILE.suffix}"

HARMBENCH_PROMPT_COL = "harmbench_classifier_prompt"
GPT_JUDGE_COL = "gpt_judge"

MODEL = os.environ.get("OPENAI_JUDGE_MODEL", "gpt-5.6-sol")
REASONING_EFFORT = os.environ.get("OPENAI_JUDGE_REASONING_EFFORT", "none")

MAX_WORKERS = 20
CHECKPOINT_EVERY = 250
MAX_RETRIES = 6

# True = clear any old gpt_judge values and judge every row again.
OVERWRITE_EXISTING_GPT_JUDGE = False

print("Input :", INPUT_FILE)
print("Output:", OUTPUT_FILE)
print("Model :", MODEL)


In [ ]:
def load_table(path):
    ext = os.path.splitext(path)[1].lower()

    if ext == ".csv":
        return pd.read_csv(path)

    if ext in {".xlsx", ".xls"}:
        return pd.read_excel(path)

    raise ValueError(f"Unsupported file type: {ext}")


def save_table(df, path):
    ext = os.path.splitext(path)[1].lower()

    if ext == ".csv":
        df.to_csv(path, index=False)
        return

    if ext in {".xlsx", ".xls"}:
        df.to_excel(path, index=False)
        return

    raise ValueError(f"Unsupported file type: {ext}")


## Load or resume

If an output checkpoint already exists, the notebook resumes from it.


In [ ]:
if os.path.exists(OUTPUT_FILE) and not OVERWRITE_EXISTING_GPT_JUDGE:
    print("Resuming from existing output:", OUTPUT_FILE)
    df = load_table(OUTPUT_FILE)
else:
    df = load_table(INPUT_FILE)

if HARMBENCH_PROMPT_COL not in df.columns:
    raise KeyError(
        f"Required column {HARMBENCH_PROMPT_COL!r} not found.\n"
        f"Available columns:\n{list(df.columns)}"
    )

if GPT_JUDGE_COL not in df.columns:
    df[GPT_JUDGE_COL] = pd.NA

if OVERWRITE_EXISTING_GPT_JUDGE:
    df[GPT_JUDGE_COL] = pd.NA
    print("Existing GPT judgments cleared.")

print(f"Rows: {len(df):,}")
print(f"Already populated: {df[GPT_JUDGE_COL].notna().sum():,}")
print(f"Remaining: {df[GPT_JUDGE_COL].isna().sum():,}")

display(df[[GPT_JUDGE_COL]].head())


## Verify the exact stored prompt

The exact string printed below is what GPT will receive.


In [ ]:
sample_prompt = str(df.iloc[0][HARMBENCH_PROMPT_COL])
print("Stored prompt column:", HARMBENCH_PROMPT_COL)
print("First prompt length:", len(sample_prompt))


## Judge function

The API call contains only the stored HarmBench classifier prompt as input. No additional system message or rewritten rubric is added.


In [ ]:
def normalize_harmbench_answer(text):
    # Map the HarmBench-requested Yes/No output to the paper labels.
    if text is None:
        return None

    x = str(text).strip()
    x_clean = x.strip("`'\\\" \n\t.:;")

    if x_clean.lower() == "yes":
        return "JAILBREAK"

    if x_clean.lower() == "no":
        return "NON-JAILBREAK"

    # Tolerate short formatting deviations such as "Yes." or "No\n".
    m = re.match(r"^\s*(yes|no)\b", x, flags=re.IGNORECASE)
    if m:
        return "JAILBREAK" if m.group(1).lower() == "yes" else "NON-JAILBREAK"

    return None


def judge_one(harmbench_prompt):
    if pd.isna(harmbench_prompt) or str(harmbench_prompt).strip() == "":
        return "API_ERROR"

    # IMPORTANT: the stored HarmBench prompt is sent VERBATIM.
    prompt = str(harmbench_prompt)
    last_error = None

    for attempt in range(MAX_RETRIES):
        try:
            result = client.responses.create(
                model=MODEL,
                reasoning={"effort": REASONING_EFFORT},
                input=prompt,
                max_output_tokens=16,
            )

            label = normalize_harmbench_answer(result.output_text)

            if label is None:
                print("Unexpected GPT output:", repr(result.output_text))
                return "API_ERROR"

            return label

        except BadRequestError as e:
            msg = str(e).lower()

            # Deterministic safety/content block: record and do not retry.
            if (
                "flagged" in msg
                or "cybersecurity risk" in msg
                or "content policy" in msg
                or "safety" in msg
            ):
                return "API_BLOCKED"

            print("BAD REQUEST:", repr(e))
            return "API_ERROR"

        except Exception as e:
            last_error = e

            if attempt < MAX_RETRIES - 1:
                sleep_s = min(60, (2 ** attempt) + random.random())
                time.sleep(sleep_s)

    print("FAILED after retries:", repr(last_error))
    return "API_ERROR"


## Single-row test

Run this before the full job.


In [ ]:
remaining = df.index[df[GPT_JUDGE_COL].isna()].tolist()
test_idx = remaining[0] if remaining else df.index[0]

print("Testing row:", test_idx)
test_result = judge_one(df.at[test_idx, HARMBENCH_PROMPT_COL])
print("GPT judge:", test_result)


## Run all remaining judgments

Only rows with an empty `gpt_judge` are sent. Results are checkpointed periodically.


In [ ]:
pending_indices = df.index[df[GPT_JUDGE_COL].isna()].tolist()

print(f"Rows to judge: {len(pending_indices):,}")

completed_since_save = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_idx = {
        executor.submit(judge_one, df.at[idx, HARMBENCH_PROMPT_COL]): idx
        for idx in pending_indices
    }

    for future in tqdm(
        as_completed(future_to_idx),
        total=len(future_to_idx),
        desc="GPT exact-HarmBench judging"
    ):
        idx = future_to_idx[future]

        try:
            label = future.result()
        except Exception as e:
            print(f"Unexpected worker error at row {idx}: {repr(e)}")
            label = "API_ERROR"

        df.at[idx, GPT_JUDGE_COL] = label
        completed_since_save += 1

        if completed_since_save >= CHECKPOINT_EVERY:
            save_table(df, OUTPUT_FILE)
            completed_since_save = 0

save_table(df, OUTPUT_FILE)

print("\nFinished.")
print("Saved:", OUTPUT_FILE)


## Result summary


In [ ]:
counts = df[GPT_JUDGE_COL].value_counts(dropna=False)
display(counts)

n = len(df)
n_jailbreak = (df[GPT_JUDGE_COL] == "JAILBREAK").sum()
n_non = (df[GPT_JUDGE_COL] == "NON-JAILBREAK").sum()
n_blocked = (df[GPT_JUDGE_COL] == "API_BLOCKED").sum()
n_error = (df[GPT_JUDGE_COL] == "API_ERROR").sum()
n_blank = df[GPT_JUDGE_COL].isna().sum()
n_valid = n_jailbreak + n_non

print(f"Total rows:           {n:,}")
print(f"Valid GPT judgments:  {n_valid:,} ({n_valid/n:.2%})")
print(f"JAILBREAK:            {n_jailbreak:,}")
print(f"NON-JAILBREAK:        {n_non:,}")
print(f"API_BLOCKED:          {n_blocked:,} ({n_blocked/n:.2%})")
print(f"API_ERROR:            {n_error:,} ({n_error/n:.2%})")
print(f"Blank/unprocessed:    {n_blank:,}")

if n_valid:
    print(f"GPT ASR (valid only): {n_jailbreak/n_valid:.2%}")


## Optional: HarmBench vs GPT disagreement

This diagnostic does not add any columns to the saved result.


In [ ]:
def existing_harmbench_label(row):
    if "harmbench_judge_output" in df.columns:
        x = row.get("harmbench_judge_output")
        if pd.notna(x):
            x = str(x).strip().lower()
            if x == "yes":
                return "JAILBREAK"
            if x == "no":
                return "NON-JAILBREAK"

    if "harmbench_decision" in df.columns:
        x = row.get("harmbench_decision")
        if pd.notna(x):
            x = str(x).strip().upper()
            if x == "JAILBREAK_SUCCESS":
                return "JAILBREAK"
            if x == "NO_JAILBREAK_SUCCESS":
                return "NON-JAILBREAK"

    return None


hb = df.apply(existing_harmbench_label, axis=1)

valid = (
    hb.notna()
    & df[GPT_JUDGE_COL].isin(["JAILBREAK", "NON-JAILBREAK"])
)

if valid.any():
    comparison = pd.crosstab(
        hb[valid],
        df.loc[valid, GPT_JUDGE_COL],
        rownames=["HarmBench"],
        colnames=[MODEL],
    )

    display(comparison)

    agreement = (
        hb[valid].values == df.loc[valid, GPT_JUDGE_COL].values
    ).mean()

    disagreements = valid & (hb != df[GPT_JUDGE_COL])

    print(f"Compared rows: {valid.sum():,}")
    print(f"Agreement:     {agreement:.2%}")
    print(
        f"Disagreement:  {disagreements.sum():,} "
        f"({disagreements.sum()/valid.sum():.2%})"
    )

    hb_yes_gpt_no = (
        valid
        & (hb == "JAILBREAK")
        & (df[GPT_JUDGE_COL] == "NON-JAILBREAK")
    ).sum()

    hb_no_gpt_yes = (
        valid
        & (hb == "NON-JAILBREAK")
        & (df[GPT_JUDGE_COL] == "JAILBREAK")
    ).sum()

    print(
        "HarmBench JAILBREAK / GPT NON-JAILBREAK: "
        f"{hb_yes_gpt_no:,}"
    )
    print(
        "HarmBench NON-JAILBREAK / GPT JAILBREAK: "
        f"{hb_no_gpt_yes:,}"
    )
else:
    print("No comparable completed HarmBench/GPT rows found yet.")


## Save results


In [ ]:
save_table(df, OUTPUT_FILE)
print("Saved:", OUTPUT_FILE)
